CELL 1 — Imports and Setup

In [ ]:
# ================================
# PHASE 3: MODEL DEVELOPMENT & TRAINING
# IRS-1: AI-Driven Demand Intelligence
# ================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import time
import os
import pickle
warnings.filterwarnings('ignore')

BASE_PATH = 'C:/Users/Dell/Desktop/Naveed/gitdemo/Forecasting-future'
MODELS_PATH = f'{BASE_PATH}/models'
os.makedirs(MODELS_PATH, exist_ok=True)

print("✅ Libraries loaded")
print(f"Models will be saved to: {MODELS_PATH}")

CELL 2 — Load Phase 2 Output

In [ ]:
# ── Load Phase 2 feature dataset ─────────────────────────────────
df = pd.read_csv(f'{BASE_PATH}/data/processed/demand_features.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['store_id', 'product_id', 'date'])
df = df.reset_index(drop=True)

print("✅ Feature dataset loaded")
print(f"Shape: {df.shape}")
print(f"Date range: {df['date'].min()} → {df['date'].max()}")
print(f"Unique stores:   {df['store_id'].nunique()}")
print(f"Unique products: {df['product_id'].nunique()}")
print(f"\nColumns: {df.columns.tolist()}")


CELL 3 — Train/Validation/Test Split

In [ ]:
# ── Temporal Split — NEVER random split for time series ──────────
# Train: 70%  Validation: 15%  Test: 15%

df_sorted = df.sort_values('date').reset_index(drop=True)
n = len(df_sorted)

train_end = int(n * 0.70)
val_end   = int(n * 0.85)

train = df_sorted.iloc[:train_end].copy()
val   = df_sorted.iloc[train_end:val_end].copy()
test  = df_sorted.iloc[val_end:].copy()

print("=" * 50)
print("TEMPORAL TRAIN/VAL/TEST SPLIT")
print("=" * 50)
print(f"Train : {len(train):,} rows "
      f"({train['date'].min().date()} → {train['date'].max().date()})")
print(f"Val   : {len(val):,} rows "
      f"({val['date'].min().date()} → {val['date'].max().date()})")
print(f"Test  : {len(test):,} rows "
      f"({test['date'].min().date()} → {test['date'].max().date()})")
print(f"\nTotal : {len(df_sorted):,} rows")
print("\n✅ No data leakage — future data never seen during training")

CELL 4 — Define Features and Target


In [ ]:
# ── Define feature columns and target ────────────────────────────
TARGET = 'sales'

# Features for ML models (XGBoost, LightGBM)
ML_FEATURES = [
    'sales_lag_1', 'sales_lag_2', 'sales_lag_3',
    'rolling_mean_2', 'rolling_std_2',
    'rolling_mean_3', 'rolling_std_3',
    'day_of_week', 'month', 'quarter',
    'week_of_year', 'is_weekend', 'year',
    'sin_1', 'cos_1', 'sin_2', 'cos_2', 'sin_3', 'cos_3',
    'promo', 'price_rolling_mean'
]

# Remove any columns that don't exist in df
ML_FEATURES = [f for f in ML_FEATURES if f in df.columns]

print("Target:", TARGET)
print(f"\nML Features ({len(ML_FEATURES)}):")
for f in ML_FEATURES:
    print(f"  {f}")

# Prepare sets
X_train = train[ML_FEATURES].fillna(0)
y_train = train[TARGET]

X_val   = val[ML_FEATURES].fillna(0)
y_val   = val[TARGET]

X_test  = test[ML_FEATURES].fillna(0)
y_test  = test[TARGET]

print(f"\nX_train: {X_train.shape}")
print(f"X_val:   {X_val.shape}")
print(f"X_test:  {X_test.shape}")
print("\n✅ Features and target ready")

CELL 5 — Evaluation Metrics Function


In [ ]:
# ── Evaluation Metrics ────────────────────────────────────────────
from sklearn.metrics import mean_absolute_error, mean_squared_error

def evaluate_model(y_true, y_pred, model_name):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    # MAPE — avoid division by zero
    mask = y_true != 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask])
                           / y_true[mask])) * 100

    # sMAPE
    smape = np.mean(2 * np.abs(y_true - y_pred) /
                    (np.abs(y_true) + np.abs(y_pred) + 1e-8)) * 100

    results = {
        'Model': model_name,
        'MAE':   round(mae, 4),
        'RMSE':  round(rmse, 4),
        'MAPE':  round(mape, 2),
        'sMAPE': round(smape, 2)
    }

    print(f"\n{'='*45}")
    print(f"  {model_name}")
    print(f"{'='*45}")
    print(f"  MAE   : {mae:.4f}")
    print(f"  RMSE  : {rmse:.4f}")
    print(f"  MAPE  : {mape:.2f}%")
    print(f"  sMAPE : {smape:.2f}%")

    return results

# Store all results here
all_results = []
print("✅ Metrics function ready")

CELL 6 — Model 1: Naive Baseline


In [ ]:
# ════════════════════════════════════════
# MODEL 1: NAIVE BASELINE
# Predict next value = last observed value
# ════════════════════════════════════════
print("Training Naive Baseline...")
start = time.time()

# Naive — just use lag_1 as prediction
naive_pred_val  = X_val['sales_lag_1'].fillna(y_train.mean())
naive_pred_test = X_test['sales_lag_1'].fillna(y_train.mean())

elapsed = time.time() - start
print(f"⏱ Time: {elapsed:.1f} seconds")

results_naive = evaluate_model(y_test, naive_pred_test, "Naive Baseline")
results_naive['Time_sec'] = round(elapsed, 2)
all_results.append(results_naive)

# Save predictions
test = test.copy()
test['pred_naive'] = naive_pred_test.values
print("\n✅ Naive Baseline complete")

CELL 7 — Model 2: ETS (Holt-Winters)


In [ ]:
# ════════════════════════════════════════
# MODEL 2: ETS — EXPONENTIAL SMOOTHING
# ════════════════════════════════════════
from statsmodels.tsa.holtwinters import SimpleExpSmoothing

print("Training ETS (Exponential Smoothing)...")
start = time.time()

# Use overall time series (aggregated daily)
daily_train = train.groupby('date')['sales'].mean().reset_index()
daily_test  = test.groupby('date')['sales'].mean().reset_index()

ets_model = SimpleExpSmoothing(daily_train['sales'].values,
                                initialization_method='estimated')
ets_fit   = ets_model.fit(optimized=True)

# Forecast for test period length
ets_forecast = ets_fit.forecast(len(daily_test))

# Map back to test rows
ets_pred_test = pd.Series(
    [ets_forecast[0]] * len(test),
    index=test.index
)

elapsed = time.time() - start
print(f"⏱ Time: {elapsed:.1f} seconds")

results_ets = evaluate_model(y_test, ets_pred_test, "ETS (Exp Smoothing)")
results_ets['Time_sec'] = round(elapsed, 2)
all_results.append(results_ets)

test['pred_ets'] = ets_pred_test.values
print("\n✅ ETS complete")

CELL 8 — Model 3: ARIMA


In [ ]:
# ════════════════════════════════════════
# MODEL 3: ARIMA
# ════════════════════════════════════════
from statsmodels.tsa.arima.model import ARIMA

print("Training ARIMA... (may take 5-10 minutes)")
start = time.time()

# Use aggregated daily sales
daily_train_vals = daily_train['sales'].values

# Auto-select order (p=1,d=1,q=1) — simple but effective
try:
    arima_model = ARIMA(daily_train_vals, order=(1, 1, 1))
    arima_fit   = arima_model.fit()
    arima_forecast = arima_fit.forecast(steps=len(daily_test))
    arima_pred_test = pd.Series(
        [arima_forecast[0]] * len(test),
        index=test.index
    )
    print("✅ ARIMA(1,1,1) converged")
except Exception as e:
    print(f"ARIMA(1,1,1) failed: {e}")
    print("Trying ARIMA(1,0,0)...")
    arima_model = ARIMA(daily_train_vals, order=(1, 0, 0))
    arima_fit   = arima_model.fit()
    arima_forecast = arima_fit.forecast(steps=len(daily_test))
    arima_pred_test = pd.Series(
        [arima_forecast[0]] * len(test),
        index=test.index
    )

elapsed = time.time() - start
print(f"⏱ Time: {elapsed:.1f} seconds")

results_arima = evaluate_model(y_test, arima_pred_test, "ARIMA")
results_arima['Time_sec'] = round(elapsed, 2)
all_results.append(results_arima)

test['pred_arima'] = arima_pred_test.values
print("\n✅ ARIMA complete")

CELL 9 — Model 4: XGBoost


In [ ]:
# ════════════════════════════════════════
# MODEL 4: XGBOOST
# ════════════════════════════════════════
import xgboost as xgb

print("Training XGBoost...")
start = time.time()

xgb_model = xgb.XGBRegressor(
    n_estimators    = 500,
    max_depth       = 6,
    learning_rate   = 0.05,
    subsample       = 0.8,
    colsample_bytree= 0.8,
    random_state    = 42,
    n_jobs          = -1,
    verbosity       = 0
)

xgb_model.fit(
    X_train, y_train,
    eval_set        = [(X_val, y_val)],
    verbose         = False
)

xgb_pred_test = xgb_model.predict(X_test)

elapsed = time.time() - start
print(f"⏱ Time: {elapsed:.1f} seconds")

results_xgb = evaluate_model(y_test, xgb_pred_test, "XGBoost")
results_xgb['Time_sec'] = round(elapsed, 2)
all_results.append(results_xgb)

test['pred_xgb'] = xgb_pred_test

# Save model
pickle.dump(xgb_model,
            open(f'{MODELS_PATH}/xgboost_model.pkl', 'wb'))
print(f"\n✅ XGBoost complete — saved to models/")

CELL 10 — Model 5: LightGBM


In [ ]:
# ════════════════════════════════════════
# MODEL 5: LIGHTGBM
# ════════════════════════════════════════
import lightgbm as lgb

print("Training LightGBM...")
start = time.time()

lgb_model = lgb.LGBMRegressor(
    n_estimators   = 500,
    max_depth      = 6,
    learning_rate  = 0.05,
    subsample      = 0.8,
    colsample_bytree=0.8,
    random_state   = 42,
    n_jobs         = -1,
    verbose        = -1
)

lgb_model.fit(
    X_train, y_train,
    eval_set = [(X_val, y_val)],
    callbacks= [lgb.early_stopping(50, verbose=False),
                lgb.log_evaluation(period=-1)]
)

lgb_pred_test = lgb_model.predict(X_test)

elapsed = time.time() - start
print(f"⏱ Time: {elapsed:.1f} seconds")

results_lgb = evaluate_model(y_test, lgb_pred_test, "LightGBM")
results_lgb['Time_sec'] = round(elapsed, 2)
all_results.append(results_lgb)

test['pred_lgb'] = lgb_pred_test

# Save model
pickle.dump(lgb_model,
            open(f'{MODELS_PATH}/lightgbm_model.pkl', 'wb'))
print(f"\n✅ LightGBM complete — saved to models/")

CELL 11 — Model 6: LSTM


In [ ]:
# ════════════════════════════════════════
# MODEL 6: LSTM
# ════════════════════════════════════════
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip',
                       'install', 'torch', '--quiet'])
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler

print("Training LSTM... (15-25 minutes on Core i5)")
start = time.time()

# ── Scale sales ───────────────────────────────────────────────────
scaler = MinMaxScaler()
y_train_scaled = scaler.fit_transform(
    y_train.values.reshape(-1, 1)).flatten()
y_val_scaled   = scaler.transform(
    y_val.values.reshape(-1, 1)).flatten()
y_test_scaled  = scaler.transform(
    y_test.values.reshape(-1, 1)).flatten()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.fit_transform(X_val)
X_test_scaled  = scaler.fit_transform(X_test)

# ── Convert to tensors ────────────────────────────────────────────
def to_tensor(X, y):
    X_t = torch.FloatTensor(X).unsqueeze(1)
    y_t = torch.FloatTensor(y)
    return TensorDataset(X_t, y_t)

train_ds = to_tensor(X_train_scaled, y_train_scaled)
val_ds   = to_tensor(X_val_scaled,   y_val_scaled)
test_ds  = to_tensor(X_test_scaled,  y_test_scaled)

train_dl = DataLoader(train_ds, batch_size=64, shuffle=False)
val_dl   = DataLoader(val_ds,   batch_size=64, shuffle=False)

# ── Define LSTM model ─────────────────────────────────────────────
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden=64, layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden,
                            num_layers=layers,
                            batch_first=True,
                            dropout=dropout)
        self.fc = nn.Linear(hidden, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :]).squeeze()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

model = LSTMModel(input_size=len(ML_FEATURES)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

# ── Training loop ─────────────────────────────────────────────────
EPOCHS = 30
best_val_loss = float('inf')
best_state    = None

for epoch in range(EPOCHS):
    model.train()
    for X_batch, y_batch in train_dl:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        pred = model(X_batch)
        loss = criterion(pred, y_batch)
        loss.backward()
        optimizer.step()

    # Validation
    model.eval()
    val_losses = []
    with torch.no_grad():
        for X_batch, y_batch in val_dl:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            pred = model(X_batch)
            val_losses.append(criterion(pred, y_batch).item())
    val_loss = np.mean(val_losses)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state    = model.state_dict().copy()

    if (epoch + 1) % 5 == 0:
        print(f"  Epoch {epoch+1}/{EPOCHS} "
              f"— val_loss: {val_loss:.6f}")

# ── Predict on test ───────────────────────────────────────────────
model.load_state_dict(best_state)
model.eval()
X_test_t = torch.FloatTensor(X_test_scaled).unsqueeze(1).to(device)

with torch.no_grad():
    lstm_pred_scaled = model(X_test_t).cpu().numpy()

lstm_pred_test = scaler.inverse_transform(
    lstm_pred_scaled.reshape(-1, 1)).flatten()

elapsed = time.time() - start
print(f"\n⏱ Time: {elapsed/60:.1f} minutes")

results_lstm = evaluate_model(y_test, lstm_pred_test, "LSTM")
results_lstm['Time_sec'] = round(elapsed, 2)
all_results.append(results_lstm)

test['pred_lstm'] = lstm_pred_test

# Save model
torch.save(best_state, f'{MODELS_PATH}/lstm_model.pt')
print(f"\n✅ LSTM complete — saved to models/")

CELL 12 — Model 7: TFT (Temporal Fusion Transformer)


In [ ]:
# ════════════════════════════════════════
# MODEL 7: TEMPORAL FUSION TRANSFORMER
# ════════════════════════════════════════
subprocess.check_call([sys.executable, '-m', 'pip',
                       'install', 'darts', '--quiet'])
from darts import TimeSeries
from darts.models import TFTModel
from darts.dataprocessing.transformers import Scaler

print("Training TFT... (45-90 minutes on Core i5)")
start = time.time()

try:
    # ── Prepare daily aggregated series ──────────────────────────
    daily = df.groupby('date')['sales'].mean().reset_index()
    daily = daily.sort_values('date').reset_index(drop=True)

    series = TimeSeries.from_dataframe(
        daily, 'date', 'sales', freq='D', fill_missing_dates=True)

    # Split
    n_test = int(len(series) * 0.15)
    train_series = series[:-n_test]
    test_series  = series[-n_test:]

    # Scale
    transformer = Scaler()
    train_scaled = transformer.fit_transform(train_series)
    test_scaled  = transformer.transform(test_series)

    # ── TFT Model ─────────────────────────────────────────────────
    tft = TFTModel(
        input_chunk_length  = 14,
        output_chunk_length = 7,
        hidden_size         = 32,
        lstm_layers         = 1,
        num_attention_heads = 2,
        dropout             = 0.1,
        batch_size          = 32,
        n_epochs            = 20,
        random_state        = 42
    )

    tft.fit(train_scaled, verbose=True)

    # Predict
    tft_pred = tft.predict(n=n_test)
    tft_pred_orig = transformer.inverse_transform(tft_pred)
    tft_values = tft_pred_orig.values().flatten()

    # Align with test set
    tft_pred_test = np.full(len(y_test),
                            tft_values.mean())

    elapsed = time.time() - start
    print(f"\n⏱ Time: {elapsed/60:.1f} minutes")

    results_tft = evaluate_model(y_test, tft_pred_test,
                                 "Temporal Fusion Transformer")
    results_tft['Time_sec'] = round(elapsed, 2)
    all_results.append(results_tft)

    test['pred_tft'] = tft_pred_test
    tft.save(f'{MODELS_PATH}/tft_model')
    print(f"\n✅ TFT complete — saved to models/")

except Exception as e:
    print(f"⚠️ TFT Error: {e}")
    print("Skipping TFT — add to results as N/A")
    all_results.append({
        'Model': 'TFT', 'MAE': None,
        'RMSE': None, 'MAPE': None,
        'sMAPE': None, 'Time_sec': None
    })

CELL 13 — Final Comparison Table


In [ ]:
# ════════════════════════════════════════
# FINAL RESULTS — ALL 7 MODELS COMPARED
# ════════════════════════════════════════
results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values('MAE')
results_df = results_df.reset_index(drop=True)
results_df.index += 1

print("=" * 65)
print("         PHASE 3 — MODEL BENCHMARKING RESULTS")
print("=" * 65)
print(results_df.to_string(index=True))
print("=" * 65)
print(f"\n🏆 Best Model (lowest MAE): "
      f"{results_df.iloc[0]['Model']}")
print(f"   MAE:   {results_df.iloc[0]['MAE']}")
print(f"   RMSE:  {results_df.iloc[0]['RMSE']}")
print(f"   MAPE:  {results_df.iloc[0]['MAPE']}%")

# Save results
results_df.to_csv(
    f'{BASE_PATH}/data/processed/model_results.csv',
    index=False
)
print(f"\n✅ Results saved → data/processed/model_results.csv")

CELL 14 — Visualize Results


In [ ]:
# ── Plot comparison ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Phase 3 — Model Comparison', fontsize=14,
             fontweight='bold')

metrics = ['MAE', 'RMSE', 'MAPE']
colors  = ['#3498DB','#E74C3C','#27AE60','#F39C12',
           '#9B59B6','#1ABC9C','#E67E22']

for i, metric in enumerate(metrics):
    vals = results_df[metric].dropna()
    mods = results_df.loc[vals.index, 'Model']
    axes[i].barh(range(len(vals)), vals.values,
                 color=colors[:len(vals)], edgecolor='white')
    axes[i].set_yticks(range(len(vals)))
    axes[i].set_yticklabels(mods, fontsize=9)
    axes[i].set_title(f'{metric} (lower is better)')
    axes[i].set_xlabel(metric)
    axes[i].grid(True, alpha=0.3, axis='x')
    axes[i].invert_yaxis()

plt.tight_layout()
plt.savefig(f'{BASE_PATH}/data/processed/phase3_results.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Chart saved → data/processed/phase3_results.png")

CELL 15 — Feature Importance (XGBoost + LightGBM)


In [ ]:
# ── Feature Importance ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Feature Importance', fontsize=14, fontweight='bold')

# XGBoost importance
xgb_imp = pd.Series(xgb_model.feature_importances_,
                    index=ML_FEATURES).sort_values(ascending=True).tail(15)
axes[0].barh(range(len(xgb_imp)), xgb_imp.values, color='#3498DB')
axes[0].set_yticks(range(len(xgb_imp)))
axes[0].set_yticklabels(xgb_imp.index, fontsize=8)
axes[0].set_title('XGBoost Feature Importance')
axes[0].set_xlabel('Importance Score')

# LightGBM importance
lgb_imp = pd.Series(lgb_model.feature_importances_,
                    index=ML_FEATURES).sort_values(ascending=True).tail(15)
axes[1].barh(range(len(lgb_imp)), lgb_imp.values, color='#27AE60')
axes[1].set_yticks(range(len(lgb_imp)))
axes[1].set_yticklabels(lgb_imp.index, fontsize=8)
axes[1].set_title('LightGBM Feature Importance')
axes[1].set_xlabel('Importance Score')

plt.tight_layout()
plt.savefig(f'{BASE_PATH}/data/processed/feature_importance.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("\n🎉 PHASE 3 COMPLETE!")
print(f"Models saved: {os.listdir(MODELS_PATH)}")

Phase 3 Summary
CELLS    WHAT IT DOES
──────   ─────────────────────────────────────
Cell 1   Imports and setup
Cell 2   Load Phase 2 demand_features.csv
Cell 3   Temporal train/val/test split (70/15/15)
Cell 4   Define features and target variable
Cell 5   Evaluation metrics function (MAE/RMSE/MAPE/sMAPE)
Cell 6   Model 1 — Naive Baseline        (~2 sec)
Cell 7   Model 2 — ETS Exponential       (~30 sec)
Cell 8   Model 3 — ARIMA                 (~5-10 min)
Cell 9   Model 4 — XGBoost               (~1-2 min)
Cell 10  Model 5 — LightGBM              (~1-2 min)
Cell 11  Model 6 — LSTM                  (~15-25 min)
Cell 12  Model 7 — TFT                   (~45-90 min)
Cell 13  Final comparison table
Cell 14  Visualization chart
Cell 15  Feature importance plots

TODAY:
  Run Cells 1-10 (Naive, ETS, ARIMA, XGBoost, LightGBM)
  Total time: ~20 minutes

BEFORE SLEEPING:
  Run Cell 11 (LSTM) — let it run while you sleep
  Run Cell 12 (TFT)  — let it run while you sleep

NEXT MORNING:
  Run Cells 13-15 to see final results